# Crop `IcelandDEM_20mI.grd` by geographic extent

This notebook:

1. reads the original grid metadata,
2. plots a downsampled preview of the full DEM,
3. plots the original geographic extent,
4. extracts a smaller lon/lat subset in memory,
5. plots the cropped DEM,
6. exports the cropped subset.

Adjust the subset bounds in the parameter cell before running the crop cells.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from netCDF4 import Dataset

dem_path = Path('/raid2/jam247/JVGR_ASKJA/gmt_data/IcelandDEM_20mI.grd')
output_path = dem_path.with_name('IcelandDEM_20mI_subset.grd')

# Slightly larger than: region=-17.3/-15.9/64.845/65.35
subset_lon_min = -17.4
subset_lon_max = -15.8
subset_lat_min = 64.8
subset_lat_max = 65.4


In [ ]:
with Dataset(dem_path) as ds:
    lon = ds.variables['x'][:]
    lat = ds.variables['y'][:]
    z = ds.variables['z']
    nx = len(lon)
    ny = len(lat)
    dx = float(lon[1] - lon[0])
    dy = float(lat[1] - lat[0])
    fill_value = np.nan
    z_min = float(np.nanmin(z[:]))
    z_max = float(np.nanmax(z[:]))

full_extent = {
    'lon_min': float(lon.min()),
    'lon_max': float(lon.max()),
    'lat_min': float(lat.min()),
    'lat_max': float(lat.max()),
    'dx': dx,
    'dy': dy,
    'nx': nx,
    'ny': ny,
    'z_min': z_min,
    'z_max': z_max,
}

full_extent

In [ ]:
# Plot a downsampled preview of the full DEM.
preview_max_dim = 1500
preview_stride = max(1, int(np.ceil(max(nx, ny) / preview_max_dim)))

with Dataset(dem_path) as ds:
    preview_data = ds.variables['z'][::preview_stride, ::preview_stride]

preview_lon = lon[::preview_stride]
preview_lat = lat[::preview_stride]

fig, ax = plt.subplots(figsize=(10, 6))
image = ax.imshow(
    preview_data,
    extent=[preview_lon.min(), preview_lon.max(), preview_lat.min(), preview_lat.max()],
    origin='lower',
    cmap='terrain',
    aspect='auto',
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Full DEM preview (every {preview_stride}th cell)')
fig.colorbar(image, ax=ax, label='Value')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(
    [full_extent['lon_min'], full_extent['lon_max'], full_extent['lon_max'], full_extent['lon_min'], full_extent['lon_min']],
    [full_extent['lat_min'], full_extent['lat_min'], full_extent['lat_max'], full_extent['lat_max'], full_extent['lat_min']],
    linewidth=2,
    label='Original DEM extent',
)
ax.plot(
    [subset_lon_min, subset_lon_max, subset_lon_max, subset_lon_min, subset_lon_min],
    [subset_lat_min, subset_lat_min, subset_lat_max, subset_lat_max, subset_lat_min],
    linewidth=2,
    label='Requested crop extent',
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Original DEM geographic extent')
ax.set_aspect('equal', adjustable='box')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
if not (full_extent['lon_min'] <= subset_lon_min < subset_lon_max <= full_extent['lon_max']):
    raise ValueError('Subset longitudes must fall inside the DEM extent.')

if not (full_extent['lat_min'] <= subset_lat_min < subset_lat_max <= full_extent['lat_max']):
    raise ValueError('Subset latitudes must fall inside the DEM extent.')

x0 = int(np.searchsorted(lon, subset_lon_min, side='left'))
x1 = int(np.searchsorted(lon, subset_lon_max, side='right'))
y0 = int(np.searchsorted(lat, subset_lat_min, side='left'))
y1 = int(np.searchsorted(lat, subset_lat_max, side='right'))

subset_lon = lon[x0:x1]
subset_lat = lat[y0:y1]

with Dataset(dem_path) as ds:
    subset_data = ds.variables['z'][y0:y1, x0:x1].astype(np.float32)

subset_summary = {
    'shape': subset_data.shape,
    'lon_min': float(subset_lon.min()),
    'lon_max': float(subset_lon.max()),
    'lat_min': float(subset_lat.min()),
    'lat_max': float(subset_lat.max()),
    'min_value': float(np.nanmin(subset_data)),
    'max_value': float(np.nanmax(subset_data)),
}

subset_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
image = ax.imshow(
    subset_data,
    extent=[subset_lon.min(), subset_lon.max(), subset_lat.min(), subset_lat.max()],
    origin='lower',
    cmap='terrain',
    aspect='auto',
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Cropped DEM preview')
fig.colorbar(image, ax=ax, label='Value')
plt.show()

In [ ]:
with Dataset(output_path, 'w', format='NETCDF4') as out_ds:
    out_ds.createDimension('x', len(subset_lon))
    out_ds.createDimension('y', len(subset_lat))

    x_var = out_ds.createVariable('x', 'f8', ('x',))
    y_var = out_ds.createVariable('y', 'f8', ('y',))
    z_var = out_ds.createVariable('z', 'f4', ('y', 'x'), fill_value=np.nan)

    x_var[:] = subset_lon
    y_var[:] = subset_lat
    z_var[:, :] = subset_data

    x_var.long_name = 'meters'
    y_var.long_name = 'meters'
    z_var.long_name = 'meters'
    z_var.actual_range = np.array([np.nanmin(subset_data), np.nanmax(subset_data)], dtype=np.float32)

output_path